In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pymupdf
from anthropic import Anthropic

import extraction
import models
import rag
import retrieval
import utils

In [3]:
# Initialize anthropic client
client = Anthropic()

In [4]:
# Get the current directory of the Jupyter notebook
working_dir = utils.get_working_directory()
pdf_document = pymupdf.open(working_dir + "/thinkpython2.pdf")

In [5]:
# Step 2: extract every page into the knowledge base (output/)
extraction.extract_pdf(
    client=client,
    pdf_document=pdf_document,
)

  0%|          | 0/244 [00:00<?, ?it/s]

page_1.json already processed
page_2.json already processed
page_3.json already processed
page_5.json already processed
page_7.json already processed
page_4.json already processed
page_10.json already processed
page_11.json already processed
page_12.json already processed
page_8.json already processed
page_14.json already processed
page_9.json already processed
page_6.json already processed
page_16.json already processed
page_13.json already processed
page_19.json already processed
page_17.json already processed
page_18.json already processed
page_20.json already processed
page_15.json already processed
page_24.json already processed
page_22.json already processed
page_25.json already processed
page_23.json already processed
page_27.json already processed
page_28.json already processed
page_29.json already processed
page_21.json already processed
page_31.json already processed
page_33.json already processed
page_30.json already processed
page_26.json already processed
page_32.json alre

In [11]:
# Step 4: retrieve documents from the knowledge base
extracted_documents = retrieval.construct_documents()


In [12]:
extracted_documents

[{'filename': 'page_197.json',
  'content': '```python\n        for card in self.cards:\n            res.append(str(card))\n        return \'\\n\'.join(res)\n```\nThis method demonstrates an efficient way to accumulate a large string: building a list of strings and then using the string method join. The built-in function str invokes the __str__ method on each card and returns the string representation.\n\nSince we invoke join on a newline character, the cards are separated by newlines. Here\'s what the result looks like:\n```python\n>>> deck = Deck()\n>>> print(deck)\nAce of Clubs\n2 of Clubs\n3 of Clubs\n...\n10 of Spades\nJack of Spades\nQueen of Spades\nKing of Spades\n```\nEven though the result appears on 52 lines, it is one long string that contains newlines.\n18.6 Add, remove, shuffle and sort\nTo deal cards, we would like a method that removes a card from the deck and returns it. The list method pop provides a convenient way to do that:\n```python\n# inside class Deck:\n\n    d

In [19]:
# Step 4 (cont.): build the search index
index = retrieval.build_index(extracted_documents)

Connected to Elasticsearch v9.3.1


In [24]:
retrieval.search("What is a list", index, num_results=5)

Score: 3.5335176
Title: Chapter 10
Lists
This chapter presents one of Python's most useful built-in types, lists. You will also learn more about objects and what can happen when you have more than one name for the same object.
10.1 A list is a sequence
Like a string, a **list** is a sequence of values. In a string, the values are characters; in a list, they can be any type. The values in a list are called **elements** or sometimes **items**.

There are several ways to create a new list; the simplest is to enclose the elements in square brackets ([ and ]):
```python
[10, 20, 30, 40]
['crunchy frog', 'ram bladder', 'lark vomit']
```
The first example is a list of four integers. The second is a list of three strings. The elements of a list don't have to be the same type. The following list contains a string, a float, an integer, and (lo!) another list:
```python
['spam', 2.0, 5, [10, 20]]
```
A list within another list is **nested**.

A list that contains no elements is called an empty li

In [21]:
# Steps 5-6: prompt build + LLM generation
rag_engine = rag.RAG(
    index=index,
    llm_client=client,
    output_type=models.RAGResponse,
)

In [22]:
response = rag_engine.invoke(
    query="What is a list?",
    max_tokens=2056,
)
print(f"Answer: {response.answer}")  # type: ignore
print(f"Found answer: {response.found_answer}")  # type: ignore
print(f"Confidence: {response.confidence}")  # type: ignore
print(f"Answer type: {response.answer_type}")  # type: ignore
print(f"Follow-up questions: {response.followup_questions}")  # type: ignore

Answer: A **list** is one of Python's built-in types: like a string, it is a **sequence of values**. The difference is that in a string the values are characters, while in a list they can be any type. The values in a list are called **elements** or **items**.

Key points from the text:

- The simplest way to create a list is to enclose the elements in square brackets (`[` and `]`):
  ```python
  [10, 20, 30, 40]
  ['crunchy frog', 'ram bladder', 'lark vomit']
  ```
  The first is a list of four integers; the second is a list of three strings.
- Elements don't have to be the same type. For example, `['spam', 2.0, 5, [10, 20]]` contains a string, a float, an integer, and another list. A list within another list is **nested**.
- A list with no elements is an **empty list**, written with empty brackets: `[]`.
- Lists can be assigned to variables:
  ```python
  >>> cheeses = ['Cheddar', 'Edam', 'Gouda']
  >>> numbers = [42, 123]
  >>> empty = []
  >>> print(cheeses, numbers, empty)
  ['Ched

In [25]:
response = rag_engine.invoke(
    query=response.followup_questions[0],
    max_tokens=2056,
)
print(f"Answer: {response.answer}")  # type: ignore
print(f"Found answer: {response.found_answer}")  # type: ignore
print(f"Confidence: {response.confidence}")  # type: ignore
print(f"Answer type: {response.answer_type}")  # type: ignore
print(f"Follow-up questions: {response.followup_questions}")  # type: ignore

Answer: **Reading a slice.** The slice operator `t[i:j]` returns a new list containing the elements from index `i` up to (but not including) index `j`:

```python
>>> t = ['a', 'b', 'c', 'd', 'e', 'f']
>>> t[1:3]
['b', 'c']
```

**Omitting indices:**
- Omit the first index → the slice starts at the beginning: `t[:4]` → `['a', 'b', 'c', 'd']`
- Omit the second index → the slice goes to the end: `t[3:]` → `['d', 'e', 'f']`
- Omit both → you get a copy of the whole list: `t[:]` → `['a', 'b', 'c', 'd', 'e', 'f']`

Because lists are mutable, making a copy with `t[:]` is often useful before performing operations that modify a list.

**Yes, you can assign to a slice.** A slice operator on the left side of an assignment updates multiple elements at once:

```python
>>> t = ['a', 'b', 'c', 'd', 'e', 'f']
>>> t[1:3] = ['x', 'y']
>>> t
['a', 'x', 'y', 'd', 'e', 'f']
```

Here elements at indices 1 and 2 (`'b'` and `'c'`) are replaced by `'x'` and `'y'`. Note the contrast with strings: strings are